# Customer Churn Prediction - Exploratory Data Analysis

This notebook performs comprehensive EDA on the customer churn dataset.

**Author**: Your Name  
**Date**: September 2026

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 2. Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/customer_churn_data.csv')

# Display first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Dataset shape
print(f"Dataset shape: {df.shape}")
print(f"Number of customers: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")

## 3. Data Overview

In [ ]:
# Info about dataset
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values!")

## 4. Target Variable Analysis

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("Churn Distribution:")
print(f"No Churn (0): {churn_counts[0]} ({churn_pct[0]:.2f}%)")
print(f"Churn (1): {churn_counts[1]} ({churn_pct[1]:.2f}%)")

In [ ]:
# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='Churn', ax=axes[0])
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['No Churn', 'Churn'])

# Pie chart
axes[1].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Percentage')

plt.tight_layout()
plt.show()

## 5. Numerical Features Analysis

In [ ]:
# Distribution of numerical features
numerical_cols = ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=axes[idx])
    axes[idx].set_title(f'{col} Distribution by Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for numerical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[idx])
    axes[idx].set_title(f'{col} by Churn Status')
    axes[idx].set_xticklabels(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

## 6. Categorical Features Analysis

In [ ]:
# Key categorical features
categorical_cols = ['Gender', 'ContractType', 'PaymentMethod', 'InternetService']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(categorical_cols):
    churn_data = df.groupby([col, 'Churn']).size().unstack()
    churn_data.plot(kind='bar', ax=axes[idx], stacked=False)
    axes[idx].set_title(f'Churn by {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
corr_matrix = df[numerical_cols + ['Churn', 'SeniorCitizen']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Key Insights

In [ ]:
# Churn rate by contract type
contract_churn = df.groupby('ContractType')['Churn'].mean() * 100
print("Churn Rate by Contract Type:")
print(contract_churn.sort_values(ascending=False))

In [ ]:
# Churn rate by tenure groups
df['TenureGroup'] = pd.cut(df['Tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12', '13-24', '25-48', '49-72'])
tenure_churn = df.groupby('TenureGroup')['Churn'].mean() * 100
print("\nChurn Rate by Tenure Group (months):")
print(tenure_churn)

In [ ]:
# Churn rate by internet service
internet_churn = df.groupby('InternetService')['Churn'].mean() * 100
print("\nChurn Rate by Internet Service:")
print(internet_churn.sort_values(ascending=False))

## 9. Summary

### Key Findings:

1. **Contract Type Impact**
   - Month-to-month contracts have highest churn rate
   - Long-term contracts (1-2 years) show better retention

2. **Tenure Correlation**
   - New customers (< 12 months) are at highest risk
   - Churn decreases significantly after 1 year

3. **Service Features**
   - Customers without tech support show higher churn
   - Fiber optic users may have different churn patterns

4. **Financial Factors**
   - Higher monthly charges correlate with increased churn risk
   - Payment method plays a role in retention

### Recommendations for Modeling:
- Focus on contract type, tenure, and service features
- Handle class imbalance in target variable
- Consider feature engineering for tenure groups
- Test both Logistic Regression and Random Forest